In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath("/home1/smaruj/ledidi_akita/"))
from utils.df_utils import load_parameter_results

In [2]:
_PROJ     = "/project2/fudenber_735/smaruj/sequence_design/ledidi_semifreddo_akita"
RESULTS_DIR = f"{_PROJ}/optimizations/boundaries_no_ctcf"

FOLDS   = [0, 1, 2, 3, 4, 5, 6, 7]
GAMMAS = ["2e3", "3.5e3", "6e3", "1e4"]

In [3]:
# Gamma sweep
df_gamma = load_parameter_results(RESULTS_DIR, "gamma", GAMMAS, folds=FOLDS)

Total windows loaded: 1420


In [4]:
df_gamma["no_edits"] = df_gamma["n_edits"] == 0

no_edits_summary = (
    df_gamma.groupby("gamma")["no_edits"]
    .agg(n_total="count", n_no_edits="sum")
    .assign(pct_no_edits=lambda x: 100 * x["n_no_edits"] / x["n_total"])
)
print(no_edits_summary.to_string())

       n_total  n_no_edits  pct_no_edits
gamma                                   
1e4        355           0           0.0
2e3        355           0           0.0
3.5e3      355           0           0.0
6e3        355           0           0.0


In [5]:
df_gamma["insul_diff"] = df_gamma["insul_score_edited"] - df_gamma["insul_score_orig"]
df_gamma["no_improvement"] = df_gamma["insul_diff"] >= 0

no_improvement_summary = (
    df_gamma.groupby("gamma")["no_improvement"]
    .agg(n_total="count", n_no_improvement="sum")
    .assign(pct_no_improvement=lambda x: 100 * x["n_no_improvement"] / x["n_total"])
)
print(no_improvement_summary.to_string())

       n_total  n_no_improvement  pct_no_improvement
gamma                                               
1e4        355                 9            2.535211
2e3        355                10            2.816901
3.5e3      355                10            2.816901
6e3        355                 8            2.253521


In [6]:
# Create a boolean column for presence of at least one CTCF
df_gamma["has_ctcf"] = df_gamma["CTCFs_num"] > 0

# Aggregate by gamma to see the counts and percentages
ctcf_summary = (
    df_gamma.groupby("gamma")["has_ctcf"]
    .agg(n_total="count", n_has_ctcf="sum")
    .assign(pct_has_ctcf=lambda x: 100 * x["n_has_ctcf"] / x["n_total"])
)

print(ctcf_summary.to_string())

       n_total  n_has_ctcf  pct_has_ctcf
gamma                                   
1e4        355         343     96.619718
2e3        355         343     96.619718
3.5e3      355         339     95.492958
6e3        355         340     95.774648


In [7]:
# Updated success condition: 
# 1. Edits were made (~no_edits)
# 2. Insulation improved (~no_improvement)
# 3. No CTCFs were detected (CTCFs_num == 0)
success_rate = (
    df_gamma.assign(
        success=(~df_gamma["no_edits"]) & 
                (~df_gamma["no_improvement"]) & 
                (df_gamma["CTCFs_num"] == 0)
    )
    .groupby("gamma")["success"]
    .agg(n_total="count", n_success="sum")
    .assign(pct_success=lambda x: 100 * x["n_success"] / x["n_total"])
)

print(success_rate.to_string())

       n_total  n_success  pct_success
gamma                                 
1e4        355         10     2.816901
2e3        355         10     2.816901
3.5e3      355         14     3.943662
6e3        355         14     3.943662


In [8]:
summary = (
    df_gamma.groupby("gamma")
    .agg(
        n_success        = ("n_edits",          "count"),
        mean_n_edits     = ("n_edits",          "mean"),
        mean_insul_diff  = ("insul_diff",        "mean"),
        mean_CTCFs       = ("CTCFs_num",         "mean"),
    )
    .round(3)
)
print(summary.to_string())

       n_success  mean_n_edits  mean_insul_diff  mean_CTCFs
gamma                                                      
1e4          355       666.961           -0.150       3.023
2e3          355       661.003           -0.151       3.000
3.5e3        355       662.394           -0.150       2.927
6e3          355       666.527           -0.151       2.966
